# Component Evaluation

Interpret the latest component-detection baseline with per-class metrics, confusion matrix, PR curves, latency benchmarks, and a comparison to the earlier 20-class baseline.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, Markdown, display

sys.path.insert(0, str(Path('..').resolve().parent))

from ml.pipeline.component_evaluate import evaluate_component_model

dataset_root = Path('../data/component_detection_seed').resolve()
summary_path = Path('../models/component_detection/eval/latest_metrics_summary.json').resolve()

# Uncomment to refresh the evaluation artifacts.
# evaluate_component_model(dataset_root=dataset_root, split='test', batch=4, imgsz=640)

summary = json.loads(summary_path.read_text(encoding='utf-8'))
summary.keys()

In [ ]:
overall_df = pd.DataFrame([
    {
        'split': summary['split'],
        'mAP@50': summary['overall']['map50'],
        'mAP@50-95': summary['overall']['map50_95'],
        'precision': summary['overall']['precision'],
        'recall': summary['overall']['recall'],
        'baseline_mAP@50': summary['baseline_comparison']['map50'],
        'delta_mAP@50': summary['baseline_comparison']['delta_map50'],
        'ratio_mAP@50': summary['baseline_comparison']['ratio_map50'],
    }
])
overall_df

In [ ]:
per_class_df = pd.DataFrame(summary['per_class']).T.reset_index(names='class')
per_class_df = per_class_df.sort_values(['f1', 'ap50'], ascending=False).reset_index(drop=True)
per_class_df

## Interpretation

- The reduced 6-class profile is materially stronger than the earlier 20-class setup.
- `ic` is the clearest current winner, which matches expectation because it is visually larger and less repetitive than dense resistor/capacitor fields.
- `other` is carrying useful signal and is currently a better tradeoff than forcing many tiny long-tail classes.
- `resistor`, `capacitor`, and `connector` still need real AOI images because the seed dataset does not match production optics or board composition closely enough.

In [ ]:
latency_df = pd.DataFrame([summary['latency']])
latency_df

## Artifact Review

In [ ]:
artifact_paths = summary['artifacts']
for label in ['confusion_matrix', 'confusion_matrix_normalized', 'pr_curve', 'precision_curve', 'recall_curve', 'f1_curve', 'f1_bar_chart']:
    display(Markdown(f'### {label.replace("_", " ").title()}'))
    display(Image(filename=artifact_paths[label], width=900))